In [0]:
%sh
pwd
ls

### Read Data Sources

In [0]:
import pyspark.sql.functions as func

# Read the dataframe
df = spark.read.table(
    'samples.bakehouse.sales_transactions'
)

# Print schema
print(df.printSchema())

# Print a sample of data
display(df)

Databricks visualization. Run in Databricks to view.

### Projections and filters

In [0]:
df_transaction = df.select(
    'dateTime',
    func.year('dateTime').alias('year'),
    func.month('dateTime').alias('month'),
    func.day('datetime').alias('day'),
    'transactionID',
    'customerID',
    'franchiseID',
    'product',
    'quantity',
    'unitPrice',
)

df_transaction_first_half = df_transaction.filter(
    func.col('day') <= 15
)

display(df_transaction_first_half)

Databricks visualization. Run in Databricks to view.

### Renaming, adding, and dropping columns

In [0]:
df_cus = spark.read.table(
    'samples.bakehouse.sales_customers'
).drop(
    'first_name',
    'last_name',
    'email_address',
    'phone_number',
    'address',
    'state',
    'continent'
)

df_cus = df_cus.withColumnRenamed(
    'postal_zip_code',
    'customerZipcode'
).withColumnRenamed(
    'city',
    'customerCity'
).withColumnRenamed(
    'country',
    'customerCountry'
)

df_cus = df_cus.withColumn(
    'customerGender',
    func.when(
        func.col('gender') == 'female',
        'F'
    ).otherwise(
        'M'
    )
).drop('gender')


display(df_cus)

### Aggregations

How Many customers do we have ?

In [0]:
print(
  df_cus.select('customerID')
    .distinct()
    .count()
)

How many male / female customers do we have

In [0]:
display(
    df_cus.groupBy(
        'customerGender'
    ).agg(
        func.countDistinct('customerID').alias('num_customers')
    )
)

How many customers per country do we have ?



In [0]:
df_cus.printSchema
display(
    df_cus.groupBy(
        'customerCountry',
        'customerGender'
    ).agg(
        func.countDistinct('customerID').alias('num_customers')
    ).orderBy(
        func.col('num_customers').desc()
    )
)

In [0]:
df_cus.printSchema()

### Joins

In [0]:
# Read franchises
df_fr = spark.read.table(
    'samples.bakehouse.sales_franchises'
)

# Select columns of interest

df_fr = df_fr.select(
    'franchiseID',
    func.col('name').alias('franchiseName'),
    func.col('city').alias('franchiseCity'),
    func.col('country').alias('franchiseCountry'),
    func.col('zipcode').alias('franchiseZipcode'),
    func.col('size').alias('franchiseSize'),
    'supplierID'
)

# Read supliers
df_sup = spark.read.table(
    'samples.bakehouse.sales_suppliers'
)

# Select columns of interest
df_sup = df_sup.select(
    'supplierID',
    func.col('name').alias('supplierName'),
    'ingredient',
    func.col('size').alias('supplierSize')
)

# Join Franchise with Supplier
df_fr_sup = df_fr.join(
    df_sup,
    on='supplierID',
    how='left'
).drop('supplierID')

display(
    df_fr_sup
)


In [0]:
df_r = df_transaction.join(df_cus,
                            on='customerID',
                            how='left'
                            ).join(df_fr_sup,
                                   on='franchiseID',
                                   how='left')
print(df_fr.count)

display(
    df_r
)


**Exercise**: Join transactions with customers, franchise and suppliers and call it **df_r**

Create tables and views

In [0]:
df_r.createOrReplaceTempView('fact_view')


In [0]:
df_r_ = df_r.withColumnRenamed(
    'city',
    'customerCity'
).withColumnRenamed(
    'gender',
    'customerGender'
)

df_r_.printSchema()

In [0]:
df_r_.createOrReplaceTempView('fact_view_3')

In [0]:
display(
    spark.sql(
        '''
        SELECT dateTime,
            product,
            quantity,
            customerCity,
            customerGender
        FROM fact_view_3
        LIMIT 10
        '''
        )
)

In [0]:
spark.sql(
    '''
     CREATE DATABASE IF NOT EXISTS sales_db
    '''
    )
spark.sql(
    '''
    CREATE TABLE IF NOT EXISTS sales_db.sales
    (dateTime TIMESTAMP,product STRING, quantity INT, ingredient STRING)
    '''
    )
df_table= df_r.select(
    'dateTime',
    'product',
    func.col('quantity').cast('int'),
    'ingredient'
)
df_table.write.mode('overwrite').saveAsTable('sales_db.sales')

### Exploratory Data Analysis

**Exercise**: What has been the city with highest sales ?

**Exercise**: What is the hour day with highest number of sales ?

**Exercise**: What are the most selling products ?

### Windowing

In [0]:
# What is the sales percentage of each product per day
from pyspark.sql.window import Window

df_dist = df_r.groupBy(
    'day',
    'product'
).agg(
    func.sum('quantity').alias('total_sales_per_day_product')
)


df_dist = df_dist.withColumn(
    'total_sales_per_day',
    func.sum(
        'total_sales_per_day_product'
    ).over(
        Window.partitionBy(
            'day'
        ).orderBy(
            'day'
        )
    )
)

df_dist = df_dist.withColumn(
    'sales_percentage',
    func.col('total_sales_per_day_product') / func.col('total_sales_per_day')
)


display(
    df_dist.select(
        'day',
        'product',
        'total_sales_per_day_product',
        'total_sales_per_day',
        'sales_percentage'
    ).orderBy(
        func.col('day'),
        func.col('product')
    )
)

Databricks visualization. Run in Databricks to view.

### User Defined Functions UDF

Allows Data Scientists and Data Engineers to define our own functions.

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

# Define UDF
def rename_size(x):
    if x == 'S':
        return 'small'
    elif x == 'M':
        return 'medium'
    elif x == 'L':
        return 'large'
    else:
        return 'Huge'
    

# Register UDF
rename_size_udf = udf(rename_size, StringType())

# Apply UDF in select
display(
    df_r.select(
        func.col('franchiseSize').alias('franchiseSize_old'),
        func.col('supplierSize').alias('supplierSize_old'),
        rename_size_udf(func.col('franchiseSize')).alias('franchiseSize'),
        rename_size_udf(func.col('supplierSize')).alias('supplierSize')
    ).orderBy(
        func.col('franchiseSize_old'),
        func.col('supplierSize_old')
    )
)


**Exercise**: What are the ingredients per product ?

### Pivoting

In [0]:

df_rtmp = df_r.select(
  'dateTime',
  'day',
  'product',
  'quantity'
).distinct()

df_rtmp = df_rtmp.withColumn(
  'hour',
  func.hour('dateTime')
)


df_pivot = df_rtmp.groupBy(
  'day',
  'hour'
).pivot(
  'product'
).agg(
  func.sum('quantity')
).fillna(0).orderBy(
  'day',
  'hour'
)

# Rename Columns
cols = [c for c in df_pivot.columns if c not in ['day', 'hour']]

for c in cols:
  df_pivot = df_pivot.withColumnRenamed(
    c,
    f'sales_{c.replace(' ', '_')}'
  )

display(df_pivot)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Exercise**: Apply logarithm to sales and represent the historgram

### Modeling

Predict next hour sales for a specific product.

In [0]:
PRODUCT = 'Pearly Pies'
p = PRODUCT.replace(' ', '_')

# Compute next hour sales as the target to predict
df_X = df_pivot.withColumn(
    'dummy',
    func.lit('1')
).withColumn(
    f"next_sales_{p}",
    func.lead(
        f"sales_{p}"
    ).over(
        Window.partitionBy(
            'dummy'
        ).orderBy(
            'day',
            'hour'
        )
    )
).drop(
    'dummy'
).dropna()

display(df_X)

In [0]:
import mlflow
import mlflow.spark
import pandas as pd
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler

(trainDF, testDF) = df_X.randomSplit([0.8, 0.2], seed=42)

TARGET = f"next_sales_{p}"
FEATURES = [c for c in df_X.columns if c != TARGET]

rf = RandomForestRegressor(
  labelCol=f"next_sales_{p}",
  maxBins=20,
  maxDepth=5,
  numTrees=50,
  seed=42
)


pipeline = Pipeline(stages=[rf])
vecAssembler = VectorAssembler(
  inputCols=FEATURES,
  outputCol="features"
)

pipeline = Pipeline(stages=[vecAssembler, rf])

# Train model
with mlflow.start_run(run_name='random-forest') as run:
  # Log params of the model
  mlflow.log_param('maxDepth', rf.getMaxDepth())
  mlflow.log_param('numTrees', rf.getNumTrees())
  
  # Log model
  model = pipeline.fit(trainDF)
  mlflow.spark.log_model(model, 'model', dfs_tmpdir='/Volumes/workspace/default/models')
  
  # Log metrics
  predictions = model.transform(testDF)
  evaluator = RegressionEvaluator(
    labelCol=TARGET,
    predictionCol="prediction"
  )
  rmse = evaluator.setMetricName("rmse").evaluate(predictions)
  r2 = evaluator.setMetricName("r2").evaluate(predictions)
  mlflow.log_metrics(
    {
      "rmse": rmse,
      "r2" : r2
    }
  )
  
  # Log artifacts
  rfModel = model.stages[-1]
  pandasDF = pd.DataFrame(
    list(
      zip(
        FEATURES,
        rfModel.featureImportances,
      )
    ),
    columns=['feature', 'importance']
  ).sort_values(by='importance', ascending=False)

  fig = pandasDF.plot(
    kind='barh',
    x='feature',
    y='importance',
    title=f"Feature importance for {p}"
  ).get_figure()

  mlflow.log_figure(fig, "feature_importance.png")


**Excercise**: Improve the existing model.

  Try new experiments with Cross-Validation
  
  Try another model
  
  Play with hyperparameters....